# GRACE H1 — Colab CPU preparation

**Điểm bắt đầu vẫn là notebooks/full-pipeline.ipynb trên máy local.** Sang notebook này sau khi đã train head trên Kaggle và có grace-h1-trained-heads.zip.

Runtime → Change runtime type → Hardware accelerator: **None (CPU)**. Notebook chỉ chạy whitening/projection bằng NumPy, candidate retrieval, Joern cho graph thiếu, reranking và đóng băng prompt. Không train encoder/head, không gọi LLM; không cần GPU, Vertex hay ADC.

Đặt ba file vào một thư mục Google Drive, ví dụ MyDrive/GRACE-H1-transfer:

1. grace-h1-cpu-inputs.zip — dataset, sáu embedding sets, candidate/graph caches và code/config.
2. grace-h1-joern-runtime.zip — đúng Joern 4.0.623, chỉ frontend C/C++ cần dùng.
3. grace-h1-trained-heads.zip — kết quả Kaggle phù hợp config/source/data.

Input CPU có cả query code/labels để giữ snapshot/checksum; prepare loại query labels trước retrieval. Không dùng label query để chọn demo. Tất cả file/runtime chạy trong /content; Drive chỉ giữ zip đầu vào/đầu ra để tránh I/O hàng nghìn file.

In [ ]:
from pathlib import Path,PurePosixPath
import os,sys,json,zipfile,hashlib,shutil,subprocess
USE_GOOGLE_DRIVE=True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    TRANSFER=Path('/content/drive/MyDrive/GRACE-H1-transfer')
else:
    # Upload the three zip files using Colab's Files pane, then use this folder.
    TRANSFER=Path('/content/transfer')
TRANSFER.mkdir(parents=True,exist_ok=True)
ROOT=Path('/content/GRACE-H1-SecurityProjection')
INPUTS=TRANSFER/'grace-h1-cpu-inputs.zip'
HEADS=TRANSFER/'grace-h1-trained-heads.zip'
JOERN=TRANSFER/'grace-h1-joern-runtime.zip'
CHECKPOINT=TRANSFER/'grace-h1-cpu-checkpoint.zip'
OUTPUT=TRANSFER/'grace-h1-cpu-prepared.zip'
for p in (INPUTS,HEADS,JOERN):
    if not p.is_file(): raise FileNotFoundError(f'Upload required bundle first: {p}')
print('Transfer:',TRANSFER,'CPU RAM GiB:',round(os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/1024**3,2))

Mounted at /content/drive
Transfer: /content/drive/MyDrive/GRACE-H1-transfer CPU RAM GiB: 12.67


## 1. Nhập snapshot và head

Bootstrap kiểm tra đường dẫn, checksum và mode trước khi dùng code. Trên cùng runtime, file trùng được reuse; file khác nội dung bị chặn để tránh trộn experiment. Khi đổi config/source, dùng runtime mới và bundle input mới.

In [ ]:
ROOT.mkdir(parents=True,exist_ok=True)
# Copy a single archive to native disk before extracting thousands of members.
local_inputs=Path('/content/grace-h1-cpu-inputs.zip')
shutil.copyfile(INPUTS,local_inputs)
with zipfile.ZipFile(local_inputs) as z:
    manifest=json.loads(z.read('bundle_manifest.json'))
    if manifest['schema']!='grace_h1_bundle_v1' or manifest['kind']!='cpu_inputs': raise ValueError('Wrong CPU inputs')
    names=[e['path'] for e in manifest['entries']]
    if len({n.casefold() for n in names})!=len(names): raise ValueError('Duplicate paths')
    if sorted(z.namelist())!=sorted(names+['bundle_manifest.json']): raise ValueError('Unexpected members')
    for e in manifest['entries']:
        name=e['path'];rel=PurePosixPath(name)
        if rel.is_absolute() or '..' in rel.parts or ':' in name or chr(92) in name: raise ValueError('Unsafe path')
        target=ROOT/name;target.resolve().relative_to(ROOT)
        if target.exists():
            h=hashlib.sha256()
            with target.open('rb') as src:
                for chunk in iter(lambda:src.read(1024*1024),b''): h.update(chunk)
            if target.stat().st_size==e['bytes'] and h.hexdigest()==e['sha256']: continue
            raise ValueError(f'Different existing input: {name}; use a new runtime')
        target.parent.mkdir(parents=True,exist_ok=True)
        temp=target.with_name(target.name+'.bundle.tmp');h=hashlib.sha256();size=0
        with z.open(name) as src,temp.open('wb') as dst:
            for chunk in iter(lambda:src.read(1024*1024),b''):
                h.update(chunk);size+=len(chunk);dst.write(chunk)
        if size!=e['bytes'] or h.hexdigest()!=e['sha256']: raise ValueError('Input checksum mismatch')
        temp.chmod(e.get('mode',0o644) & 0o777);os.replace(temp,target)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from grace_h1.portable import import_bundle,create_bundle
from grace_h1.io import read_json
print(import_bundle(ROOT,HEADS,'trained_heads'))
print(import_bundle(ROOT,JOERN,'joern_runtime'))
if CHECKPOINT.exists(): print(import_bundle(ROOT,CHECKPOINT,'cpu_checkpoint'))
print('Inputs, heads and optional checkpoint imported')

{'kind': 'trained_heads', 'copied': 11, 'reused_identical': 0, 'archive_sha256': 'c10fe71837d59a331105a9329c450705e210763d51fa2c9cffe3e2f23cba056f'}
{'kind': 'joern_runtime', 'copied': 219, 'reused_identical': 0, 'archive_sha256': '13637b0450c970b1f0a5b41acae5fbfaad2982956821158ed40161b157b1e6e7'}
Inputs, heads and optional checkpoint imported


## 2. Dependency CPU và Java

Chỉ cài requirements-cpu.txt: không tải PyTorch/CUDA/CodeT5 hay SDK gọi LLM. Cài Java 21 như môi trường Joern nguồn. JOERN_HOME là biến môi trường; giữ pipeline.joern_home=null trong config để fingerprint không phụ thuộc đường dẫn máy.

graph_workers mặc định 2, heap mỗi JVM 2 GB. Nếu cần giảm workers, sửa config ở local và tạo lại input trước khi chạy cả hai arm. Colab không bảo đảm lượng RAM/thời gian cố định; checkpoint zip được lưu mỗi 500 graph và cuối mỗi arm.

In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements-cpu.txt')])
subprocess.check_call(['apt-get','update','-qq'])
subprocess.check_call(['apt-get','install','-y','-qq','openjdk-21-jre-headless'])
java_homes=sorted(Path('/usr/lib/jvm').glob('java-21-openjdk-*'))
if not java_homes: raise RuntimeError('Java 21 install did not provide JAVA_HOME')
os.environ['JAVA_HOME']=str(java_homes[0])
os.environ['PATH']=str(java_homes[0]/'bin')+os.pathsep+os.environ['PATH']
os.environ['JOERN_HOME']=str(ROOT/'tools'/'joern'/'joern-cli')
subprocess.check_call([str(java_homes[0]/'bin'/'java'),'-version'])
config=read_json(ROOT/'configs'/'experiment.json')
if config['pipeline']['joern_home'] is not None: raise ValueError('Set pipeline.joern_home=null in the local config and recreate CPU inputs')
from grace_h1.offload import require_trained_heads
heads=require_trained_heads(ROOT,config)
print('Verified heads; datasets:',config['datasets'],'splits:',config['splits'],'arms:',config['llm_arms'])

Verified heads; datasets: ['devign'] splits: ['test'] arms: ['baseline_whitening', 'h1_supcon_hard']


## 3. Chạy preparation — CPU/RAM

Đọc embedding cache, fit whitening trên train, apply head có sẵn, common top50 → top5 → graph → lexical/AST → prompt. Stage này không gọi inference.

Sau mỗi 500 graph đã xử lý và mỗi arm, ghi checkpoint cache mới cùng các run đã freeze về Drive. Nếu bấm Interrupt, finally cố ghi thêm checkpoint. Nếu runtime bị xóa đột ngột, dùng checkpoint Drive cuối cùng ở lần chạy sau; phần từ checkpoint đến lúc bị xóa phải làm lại. Không chạy hai notebook đồng thời với cùng TRANSFER.

In [ ]:
from grace_h1.experiment import prepare_all
import pandas as pd

def save_checkpoint():
    info=create_bundle(ROOT,ROOT/'bundles'/'grace-h1-cpu-checkpoint.zip','cpu_checkpoint')
    temp=CHECKPOINT.with_name(CHECKPOINT.name+'.tmp')
    shutil.copyfile(info['path'],temp);os.replace(temp,CHECKPOINT)
    print('Checkpoint saved:',CHECKPOINT,'bytes:',info['bytes'],flush=True)

try:
    runs=prepare_all(ROOT,config,checkpoint_callback=save_checkpoint)
finally:
    save_checkpoint()
display(pd.DataFrame(runs))

devign/test common candidates 250/2732 (0s)
devign/test common candidates 500/2732 (0s)
devign/test common candidates 750/2732 (0s)
devign/test common candidates 1000/2732 (0s)
devign/test common candidates 1250/2732 (0s)
devign/test common candidates 1500/2732 (0s)
devign/test common candidates 1750/2732 (0s)
devign/test common candidates 2000/2732 (0s)
devign/test common candidates 2250/2732 (0s)
devign/test common candidates 2500/2732 (0s)
devign/baseline_whitening: 12198 unique graph features required
graphs 500/12198
Checkpoint saved: /content/drive/MyDrive/GRACE-H1-transfer/grace-h1-cpu-checkpoint.zip bytes: 29495349
graphs 1000/12198
graphs 1500/12198
graphs 2000/12198
graphs 2500/12198
graphs 3000/12198
graphs 3500/12198
Checkpoint saved: /content/drive/MyDrive/GRACE-H1-transfer/grace-h1-cpu-checkpoint.zip bytes: 29534084
graphs 4000/12198
Checkpoint saved: /content/drive/MyDrive/GRACE-H1-transfer/grace-h1-cpu-checkpoint.zip bytes: 29862278
graphs 4500/12198
Checkpoint saved: /

,dataset,split,arm,path
0,devign,test,baseline_whitening,runs/devign/test/baseline_whitening/9afe326872...
1,devign,test,h1_supcon_hard,runs/devign/test/h1_supcon_hard/7c4a52705268af6f


## 4. Xuất kết quả hoàn tất và quay về local

cpu-prepared chứa prompts/retrieval/freeze/run selection, vector/candidate cache mới và graph mới phát sinh. Không đóng gói lại graph gốc đã nằm trong input; không có response hay credential. Notebook kiểm tra đủ mọi arm trước khi xuất.

Tải grace-h1-cpu-prepared.zip về bundles/ của project local. Giữ grace-h1-trained-heads.zip tại đó. Quay lại full-pipeline.ipynb và Run All: notebook nhận các bundle, bỏ qua training/preparation local, rồi mới preflight → gọi Vertex → evaluation.

In [ ]:
from grace_h1.offload import require_prepared_runs
print('Frozen runs ready:',require_prepared_runs(ROOT,config))
info=create_bundle(ROOT,ROOT/'bundles'/'grace-h1-cpu-prepared.zip','cpu_prepared')
temp=OUTPUT.with_name(OUTPUT.name+'.tmp')
shutil.copyfile(info['path'],temp);os.replace(temp,OUTPUT)
print('DOWNLOAD THIS:',OUTPUT,'bytes:',info['bytes'],'sha256:',info['sha256'])
from IPython.display import FileLink,display
display(FileLink(str(OUTPUT)))
# If USE_GOOGLE_DRIVE=False, download from the Files pane before disconnecting.